In [3]:
import pandas as pd
from pathlib import Path
from typing import Union

In [2]:
# Display all rows and columns
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

In [4]:
COLUMNS = [
    "date",
    "temps",
    "magasin",
    "produit",
    "cout",
    "paiement",
]


def load(filepath: Union[str, Path]) -> pd.DataFrame:
    """
    Load a tab-separated dataset with predefined column names.

    Args:
        filepath (Union[str, Path]): Path to the dataset file.

    Returns:
        pd.DataFrame: Loaded dataset.
    """
    filepath = Path(filepath)

    if not filepath.exists():
        raise FileNotFoundError(f"File not found: {filepath}")

    df = pd.read_csv(
        filepath,
        sep="\t",
        names=COLUMNS,
        header=None,
        encoding="utf-8",
    )

    return df


In [101]:
filepath = Path("datasets/purchases.txt").resolve()

print("local_filepath:",str(filepath))

local_filepath: /opt/hadoop/workspace/datasets/purchases.txt


In [61]:
# load dataset
df = load(filepath)


In [62]:

df.head()

,date,temps,magasin,produit,cout,paiement
0,2012-01-01,09:00,San Jose,Men's Clothing,214.05,Amex
1,2012-01-01,09:00,Fort Worth,Women's Clothing,153.57,Visa
2,2012-01-01,09:00,San Diego,Music,66.08,Cash
3,2012-01-01,09:00,Pittsburgh,Pet Supplies,493.51,Discover
4,2012-01-01,09:00,Omaha,Children's Clothing,235.63,MasterCard


In [63]:
# print 
df.tail(5)

,date,temps,magasin,produit,cout,paiement
4138471,2012-12-31,17:59,Albuquerque,Toys,345.70,MasterCard
4138472,2012-12-31,17:59,Rochester,DVDs,399.57,Amex
4138473,2012-12-31,17:59,Greensboro,Baby,277.27,Discover
4138474,2012-12-31,17:59,Arlington,Women's Clothing,134.95,MasterCard
4138475,2012-12-31,17:59,Corpus Christi,DVDs,441.61,Discover


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4138476 entries, 0 to 4138475
Data columns (total 6 columns):
 #   Column    Dtype  
---  ------    -----  
 0   date      str    
 1   temps     str    
 2   magasin   str    
 3   produit   str    
 4   cout      float64
 5   paiement  str    
dtypes: float64(1), str(5)
memory usage: 189.4 MB


In [102]:
df.date=pd.to_datetime(df.date)

In [103]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4138476 entries, 0 to 4138475
Data columns (total 6 columns):
 #   Column    Dtype         
---  ------    -----         
 0   date      datetime64[us]
 1   temps     str           
 2   magasin   str           
 3   produit   str           
 4   cout      float64       
 5   paiement  str           
dtypes: datetime64[us](1), float64(1), str(4)
memory usage: 189.4 MB


In [99]:
# total nnumber of stores (magasin)
print("total_stores:", df.magasin.nunique())

total_stores: 103


In [107]:
# total number of product (produit)
print("total_product:", df.produit.nunique())

total_product: 18


In [110]:
# select one magasin
df[df.magasin=='Philadelphia'].head(10)

,date,temps,magasin,produit,cout,paiement
41,2012-01-01,09:01,Philadelphia,DVDs,351.31,Cash
99,2012-01-01,09:04,Philadelphia,Pet Supplies,332.48,Amex
107,2012-01-01,09:04,Philadelphia,Women's Clothing,482.97,MasterCard
228,2012-01-01,09:10,Philadelphia,Baby,259.14,Visa
343,2012-01-01,09:16,Philadelphia,Health and Beauty,275.79,Discover
790,2012-01-01,09:37,Philadelphia,Sporting Goods,458.27,Amex
844,2012-01-01,09:40,Philadelphia,Music,223.40,MasterCard
845,2012-01-01,09:40,Philadelphia,Children's Clothing,134.46,Cash
868,2012-01-01,09:41,Philadelphia,Books,357.31,MasterCard
918,2012-01-01,09:43,Philadelphia,Children's Clothing,358.94,Visa


In [106]:
# Calculer les ventes (cout) par magasin
( df.groupby("magasin")["cout"]
        .sum()
        .reset_index()
        .rename(columns={"cout": "total_ventes"})
        .sort_values(by="total_ventes", ascending=False)
)

,magasin,total_ventes
71,Philadelphia,10190080.26
27,Durham,10153890.21
47,Laredo,10144604.98
64,Newark,10144052.80
19,Cincinnati,10139505.74
100,Washington,10139363.39
43,Irving,10133944.08
29,Fort Wayne,10132594.02
9,Baton Rouge,10131273.23
81,Sacramento,10123468.18


In [ ]:
# Calculer les ventes par categorie de produit


# II - Pyspark 

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType ,StructField,  StringType, DoubleType, DateType
from pyspark.sql.functions import sum, col

# Create a Spark session

**SparkSession** : The entry point to programming Spark with the Dataset and DataFrame API.

A SparkSession can be used to **create DataFrame**, **register DataFrame as tables**, **execute SQL over tables**, **cache tables**, and **read parquet files**

In [16]:
# create sparksession
spark = (
    SparkSession.builder
        #.master("local")
        .appName("Analysis")
        .getOrCreate()
)


In [17]:
spark.sparkContext

<SparkContext master=local[*] appName=Analysis>

In [38]:
filepath

PosixPath('/opt/hadoop/workspace/datasets/purchases.txt')

In [73]:
f"file://{filepath}"

'file:///opt/hadoop/workspace/datasets/purchases.txt'

In [95]:
schema = StructType([
    StructField("date", DateType(), True),
    StructField("temps", StringType(), True),
    StructField("magasin", StringType(), True),
    StructField("produit", StringType(), True),
    StructField("cout", DoubleType(), True),
    StructField("paiement", StringType(), True),
])

# Loads a csv or txt file 
spark_df1 = (
    spark.read
    .schema(schema)   
    .option("sep", "\t")
    .option("header", False)   
    .csv(f"file://{filepath}")
    .toDF(
        "date",
        "temps",
        "magasin",
        "produit",
        "cout",
        "paiement"
    )
)


In [96]:
spark_df1.show()

+----------+-----+--------------+--------------------+------+----------+
|      date|temps|       magasin|             produit|  cout|  paiement|
+----------+-----+--------------+--------------------+------+----------+
|2012-01-01|09:00|      San Jose|      Men's Clothing|214.05|      Amex|
|2012-01-01|09:00|    Fort Worth|    Women's Clothing|153.57|      Visa|
|2012-01-01|09:00|     San Diego|               Music| 66.08|      Cash|
|2012-01-01|09:00|    Pittsburgh|        Pet Supplies|493.51|  Discover|
|2012-01-01|09:00|         Omaha| Children's Clothing|235.63|MasterCard|
|2012-01-01|09:00|      Stockton|      Men's Clothing|247.18|MasterCard|
|2012-01-01|09:00|        Austin|             Cameras| 379.6|      Visa|
|2012-01-01|09:00|      New York|Consumer Electronics| 296.8|      Cash|
|2012-01-01|09:00|Corpus Christi|                Toys| 25.38|  Discover|
|2012-01-01|09:00|    Fort Worth|                Toys|213.88|      Visa|
|2012-01-01|09:00|     Las Vegas|         Video Gam

In [97]:
spark_df1.printSchema()


root
 |-- date: date (nullable = true)
 |-- temps: string (nullable = true)
 |-- magasin: string (nullable = true)
 |-- produit: string (nullable = true)
 |-- cout: double (nullable = true)
 |-- paiement: string (nullable = true)



In [134]:
# Calculer les ventes (cout) par magasin
sales_by_store = (
    spark_df1
    .groupBy("magasin")
    .agg(
        sum("cout").alias("total_ventes")
    )
    .orderBy(
        col("total_ventes").desc()
    )
)



In [135]:
sales_by_store.show()

[Stage 38:====>                                                   (1 + 11) / 12]

+--------------+--------------------+
|       magasin|        total_ventes|
+--------------+--------------------+
|  Philadelphia|1.0190080260000013E7|
|        Durham|1.0153890209999997E7|
|        Laredo|1.0144604979999991E7|
|        Newark|1.0144052800000003E7|
|    Cincinnati|1.0139505740000002E7|
|    Washington|1.0139363389999997E7|
|        Irving|1.0133944080000008E7|
|    Fort Wayne|1.0132594020000003E7|
|   Baton Rouge|1.0131273230000002E7|
|    Sacramento|1.0123468180000002E7|
|    Fort Worth|1.0120830649999995E7|
| Oklahoma City|1.0118986250000006E7|
|     Charlotte|1.0112531339999998E7|
|         Tampa|1.0106428550000008E7|
|     Baltimore|1.0096521450000003E7|
|  Indianapolis|1.0090272770000005E7|
|    Pittsburgh|       1.009012482E7|
|       Norfolk|1.0088563170000002E7|
|Virginia Beach|1.0086553499999994E7|
|      New York|1.0085293550000003E7|
+--------------+--------------------+
only showing top 20 rows


In [ ]:
# Write the result to a CSV file 
sales_by_store.write.csv("hdfs:///spark/sales_by_store.csv", header=True)

In [ ]:
# Calculer les ventes par categorie de produit



In [ ]:
# Write the result to a CSV file